# DataInsight — Time series: ✏️ <project title>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trthminh/CO5177/blob/main/notebooks/timeseries.ipynb)

| | |
|---|---|
| **Course** | Programming Foundations for Data Analysis and Visualization (CO5177) — Semester 261, 2026–2027 |
| **Lecturer** | Lê Thành Sách — Faculty of Computer Science and Engineering, HCMUT – VNU-HCM |
| **Group** | DataInsight — Trương Thanh Minh (2580893), Tô Anh Phát (2580894), Trần Lê Hoàng Long (2570443) |
| **Owner of this notebook** | Trương Thanh Minh (2580893) |
| **Dataset** | ✏️ [<dataset name>](<link>) — <source>, <license> |
| **Task** | ✏️ Forecast `<target>` `<HORIZON>` step(s) ahead from its history, the calendar and covariates |

**Problem statement.** ✏️ One paragraph: what is predicted, from which inputs, why it matters, and which metric decides the best model.

**Table of contents**
1. [Setup](#s1)
2. [Data loading](#s2)
3. [Dataset description](#s3)
4. [Data quality checks](#s4)
5. [Exploratory data analysis (EDA)](#s5)
6. [Data preparation](#s6)
7. [Modeling](#s7)
8. [Evaluation on the test set](#s8)
9. [Inference on new data](#s9)
10. [Conclusion](#s10)

> **How to run.** Open in Google Colab → **Runtime → Run all**. The notebook installs missing libraries and downloads the data by itself; no manual upload, Drive mount or API key is needed.

> ### 🧭 How to use this template — delete this cell before submitting
> 1. Copy this file to `notebooks/` with its final name (the Colab badge above already points there) and keep the section numbers and headings unchanged, so the three notebooks of the group read the same way.
> 2. Fill in **section 1.3 (Configuration)** first. The notebook refuses to run while a `"TODO"` value is left there.
> 3. Run all once. Every generic cell (missing values, duplicates, split, pipelines, metrics) adapts to your columns by itself.
> 4. Then work through every `✏️ TODO` (markdown) and `# TODO` (code): dataset-specific rules, EDA questions, observations with **numbers from your own output**.
> 5. Cell pattern used everywhere: **markdown "what and why" → code with comments → markdown "Observation / Conclusion"**. Never leave a figure without a written conclusion.
> 6. Before submitting, go through the checklist in section 10 and `notebooks/templates/README.md`.
>
> Template source: `notebooks/templates/timeseries_template.ipynb`.

<a id="s1"></a>
## 1. Setup

### 1.1 Install missing libraries

Google Colab already ships most of these libraries. The cell below only installs a package when Python cannot find it, so Run all stays fast on Colab and still works on a fresh local environment. `statsmodels` ≥ 0.14 is needed for `MSTL` (several seasonal cycles at once).

In [ ]:
import importlib.util
import subprocess
import sys

# import name -> pip requirement. Add a line here for every extra library you use.
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.2",
    "plotly": "plotly>=5.15",
    "gdown": "gdown>=4.6",
    "kagglehub": "kagglehub",
    "statsmodels": "statsmodels>=0.14",
}

# find_spec returns None when a module is not installed; nothing is imported yet, so no restart is needed.
missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

### 1.2 Imports

In [ ]:
import itertools
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import MSTL, STL           # seasonal-trend decomposition (several / one cycle)
from statsmodels.tsa.stattools import acf, pacf          # autocorrelation functions

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge                   # linear regression with L2 penalty: stable with correlated lags
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit  # CV folds that always train on the past
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

### 1.3 Configuration

**Everything that depends on the chosen dataset is set here and only here.** Periods, lags and windows are counted in **steps of `FREQ`** (with hourly data, 24 steps = 1 day).

In [ ]:
# ---- Reproducibility --------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)

# ---- Data source (section 2) ------------------------------------------------------------------
DATA_DIR = Path("data")
DATA_SOURCE = {"kind": "TODO", "location": "TODO", "filename": "TODO.csv"}  # see the table in section 2.1
READ_KWARGS = {}

# ---- Series definition ------------------------------------------------------------------------
TIME_COL = "TODO"              # timestamp column; if the time is split over columns, edit build_timestamp (2.2)
TARGET = "TODO"                # numeric value to forecast
FREQ = "TODO"                  # pandas frequency of the series: "h" hourly, "D" daily, "W" weekly, "MS" monthly
NUM_EXOG = []                  # numeric covariates (weather, price, ...)
CAT_EXOG = []                  # categorical covariates (weather type, holiday name, ...)
DUPLICATE_AGG = "mean"         # how rows sharing one timestamp are merged (numeric columns)
WINDOW_START, WINDOW_END = None, None  # optional period to keep, e.g. "2016-01-01", decided in section 4.2

# ---- Forecasting setup ------------------------------------------------------------------------
HORIZON = 1                    # forecast this many steps ahead; only information >= HORIZON steps old is used
SEASONAL_PERIODS = [24, 168]   # TODO: cycles in steps (hourly: 24 = day, 168 = week; daily: 7, 365; monthly: 12)
LAGS = [1, 2, 3, 24, 168]      # TODO: chosen from the ACF/PACF in section 5.6
ROLL_WINDOWS = [24]            # rolling mean/std windows (in steps) over the past
MAX_INTERP_GAP = 3             # gaps up to this many steps are interpolated; longer gaps stay missing
TRAIN_END, VAL_END = "TODO", "TODO"   # last timestamp of train and of validation, e.g. "2017-11-30 23:00"
VALID_RANGES = {}              # TODO: physically impossible values, e.g. {"temp": (-60, 60)}; None = no bound
EVAL_ON_OBSERVED_ONLY = True   # score models only on time steps that were observed, not interpolated

# Stop Run all early with a clear message instead of failing 20 cells later.
for _name, _value in {"DATA_SOURCE": DATA_SOURCE, "TARGET": TARGET, "FREQ": FREQ, "TRAIN_END": TRAIN_END, "VAL_END": VAL_END}.items():
    assert "TODO" not in str(_value), f"Fill in {_name} in section 1.3 before running the notebook."

LAGS = sorted(k for k in LAGS if k >= HORIZON)  # a lag shorter than the horizon is not known at forecast time
print("Lags used:", LAGS)

### 1.4 Plot style and shared helpers

All figures use **Plotly** (interactive: hover shows exact values, zoom works on long series). One shared template keeps colours and fonts identical across the three notebooks of the group — the palette is the same as in the Time series notebook.

Rules for every figure:
- Build it with `plotly.express` / `plotly.graph_objects`, then call `show_fig(fig, title)`. The helper numbers figures automatically (`Figure 1.`, `Figure 2.`, …), so numbering never breaks when a cell is added.
- The title states the **conclusion** ("Class 1 is only 12% of samples"), not the chart type ("Bar chart of classes").
- Label both axes with units, and follow every figure with a markdown observation that quotes numbers.

In [ ]:
# Colours by role, fixed order: categorical series take BLUE, ORANGE, AQUA, YELLOW, then RED/MUTED.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID = "#e34948", "#898781", "#e1e0d9"
QUALITATIVE = [BLUE, ORANGE, AQUA, YELLOW, RED, MUTED]
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # counts, intensities
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]                                          # correlations (-1..1)

# A named Plotly template = one place to change the look of every chart.
pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=QUALITATIVE,
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=70, b=50),
))
pio.templates.default = "plotly_white+datainsight"

_figure_number = itertools.count(1)  # global counter: Run all numbers figures 1, 2, 3, ... in order


def show_fig(fig, title, height=420, **layout):
    """Give the figure its number and conclusion-style title, then display it."""
    fig.update_layout(title=f"Figure {next(_figure_number)}. {title}", height=height, **layout)
    fig.show()


def hist_bar(values, bins=40, color=BLUE, name=None):
    """Histogram pre-computed with NumPy: the notebook stores ~40 bars instead of every data point."""
    values = np.asarray(values, dtype=float)
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.2f}: %{y:,}<extra></extra>")


def grid_position(i, ncols):
    """(row, col) in a make_subplots grid for the i-th panel, filled left to right."""
    return i // ncols + 1, i % ncols + 1

<a id="s2"></a>
## 2. Data loading

### 2.1 Download

The data is **never uploaded by hand**: the notebook fetches it from a public source and caches it in `data/` (ignored by git), so the lecturer only has to press Run all. `download_dataset` supports three kinds of source:

| `kind` | `location` | Library | Notes |
|---|---|---|---|
| `"url"` | direct download link | `urllib` (standard library) | GitHub raw, UCI, Hugging Face `resolve/...` links, … |
| `"gdrive"` | Google Drive **file id** (the part after `/d/` in the share link) | `gdown` | Share the file as *Anyone with the link → Viewer* |
| `"kaggle"` | dataset handle `owner/dataset-name` | `kagglehub` | Public datasets need no API key |

If the link points to a `.zip` archive, also set `"archive"`; the archive is extracted and `filename` is searched inside it.

In [ ]:
def download_dataset(source, data_dir=DATA_DIR):
    """Download one data source (skipped if already cached) and return the path of `source["filename"]`."""
    data_dir.mkdir(parents=True, exist_ok=True)
    kind, location, filename = source["kind"], source["location"], source["filename"]

    if kind == "kaggle":
        import kagglehub
        root = Path(kagglehub.dataset_download(location))  # kagglehub keeps its own cache
    else:
        root = data_dir
        archive = source.get("archive")
        target = data_dir / (archive or filename)
        if not target.exists():  # cache: never download twice
            if kind == "url":
                urllib.request.urlretrieve(location, target)
            elif kind == "gdrive":
                import gdown
                gdown.download(id=location, output=str(target), quiet=False)
            else:
                raise ValueError(f"Unknown source kind {kind!r}: use 'url', 'gdrive' or 'kaggle'.")
        if archive:
            with zipfile.ZipFile(target) as zf:
                zf.extractall(data_dir)

    matches = sorted(root.rglob(filename))  # rglob: the file may sit in a sub-folder of the archive
    if not matches:
        found = [p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()][:20]
        raise FileNotFoundError(f"{filename!r} not found under {root}. Files found: {found}")
    print(f"Data file: {matches[0]} ({matches[0].stat().st_size / 1e6:,.1f} MB)")
    return matches[0]


def read_table(path, na_values=None, **kwargs):
    """Read a data file with the pandas reader that matches its extension."""
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt", ".data"}:
        return pd.read_csv(path, na_values=na_values, **kwargs)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t", na_values=na_values, **kwargs)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, na_values=na_values, **kwargs)
    if suffix == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl", **kwargs)
    raise ValueError(f"No reader configured for {suffix!r}; add one in read_table.")

### 2.2 Read the table and build the timestamp

In [ ]:
def build_timestamp(df):
    """Return one datetime per row. Unparseable values become NaT and are counted in section 4.1."""
    # TODO: if the time is split over several columns, use e.g.
    # return pd.to_datetime(df[["year", "month", "day", "hour"]], errors="coerce")
    return pd.to_datetime(df[TIME_COL], errors="coerce")


data_path = download_dataset(DATA_SOURCE)
raw = read_table(data_path, **READ_KWARGS)
raw.columns = raw.columns.str.strip()
raw["timestamp"] = build_timestamp(raw)
assert TARGET in raw.columns, f"TARGET {TARGET!r} not found. Columns: {list(raw.columns)}"

summary = pd.Series({
    "rows": f"{len(raw):,}",
    "first timestamp": raw["timestamp"].min(),
    "last timestamp": raw["timestamp"].max(),
    "time span": raw["timestamp"].max() - raw["timestamp"].min(),
    "expected steps at FREQ": f"{len(pd.date_range(raw['timestamp'].min(), raw['timestamp'].max(), freq=FREQ)):,}",
}, name="value")
display(summary.to_frame())
raw.head()

<a id="s3"></a>
## 3. Dataset description

✏️ **TODO:** Who measures the series, where, how (sensor, aggregation), time zone, licence. What is one row?

| Column | Type | Meaning | Unit |
|---|---|---|---|
| ✏️ `timestamp` | datetime | … | local time? UTC? |
| ✏️ `target` | numeric | **value to forecast** | … |

In [ ]:
raw.info()

In [ ]:
raw[[TARGET, *NUM_EXOG]].describe().T  # min/max reveal impossible values (e.g. 0 K temperature)

**Observation.** ✏️ Period covered, resolution, covariates available.

<a id="s4"></a>
## 4. Data quality checks

A regular time series needs **exactly one row per time step and no missing step**: lags (`shift`) and rolling windows count rows, not time. This section diagnoses; fixes are applied in section 6.1.

### 4.1 Timestamps: parse errors, order, duplicates

In [ ]:
ts_valid = raw["timestamp"].notna()
dup_mask = raw["timestamp"].duplicated(keep=False) & ts_valid
groups_differing = raw[dup_mask].groupby("timestamp")[TARGET].nunique().gt(1)

timestamp_checks = pd.DataFrame({
    "check": ["unparseable timestamps (NaT)", "rows already in time order",
              "extra rows sharing a timestamp", f"duplicated timestamps whose {TARGET} differs"],
    "result": [int((~ts_valid).sum()), bool(raw.loc[ts_valid, "timestamp"].is_monotonic_increasing),
               int(raw.loc[ts_valid, "timestamp"].duplicated().sum()), int(groups_differing.sum())],
})
display(timestamp_checks)
print("Most common steps between consecutive timestamps:")
raw.loc[ts_valid, "timestamp"].drop_duplicates().sort_values().diff().value_counts().head(5)

**Observation.** ✏️ Are duplicates exact copies or different measurements (→ `DUPLICATE_AGG`)? Does the most common step match `FREQ`?

### 4.2 Missing time steps

`to_regular` merges duplicates and then `asfreq(FREQ)` builds the **complete calendar**: every expected time step gets a row, and steps that were never recorded appear as `NaN`. A *gap* is a run of consecutive missing steps.

In [ ]:
def to_regular(df):
    """One row per timestamp (duplicates merged), reindexed on the full calendar at FREQ."""
    agg = {TARGET: DUPLICATE_AGG, **{c: DUPLICATE_AGG for c in NUM_EXOG}, **{c: "first" for c in CAT_EXOG}}
    out = df.dropna(subset=["timestamp"]).groupby("timestamp").agg(agg).sort_index()
    return out.asfreq(FREQ)


def gap_table(missing):
    """Start, end and length of every run of consecutive missing steps, longest first."""
    run_id = missing.ne(missing.shift()).cumsum()[missing]
    steps = pd.Series(missing.index[missing], index=run_id.values)
    return (steps.groupby(level=0).agg(start="min", end="max", steps="size")
                 .sort_values("steps", ascending=False).reset_index(drop=True))


regular = to_regular(raw)
is_missing = regular[TARGET].isna()
gaps = gap_table(is_missing)
print(f"{len(regular):,} expected steps | {is_missing.sum():,} missing ({is_missing.mean():.1%}) | {len(gaps):,} gaps")
gaps.head(10)

In [ ]:
COVERAGE_FREQ = "MS"  # TODO: period used to summarise coverage ("MS" = month start, "YS" = year, "W" = week)
coverage = (~is_missing).resample(COVERAGE_FREQ).mean() * 100
fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35], horizontal_spacing=0.1,
                    subplot_titles=("Share of steps observed per period", "Gap lengths"))
fig.add_trace(go.Bar(x=coverage.index, y=coverage.values, marker_color=BLUE, showlegend=False,
                     hovertemplate="%{x|%Y-%m-%d}: %{y:.1f}% observed<extra></extra>"), row=1, col=1)
if len(gaps):
    bins = [0, 1, MAX_INTERP_GAP, 10 * MAX_INTERP_GAP, np.inf]
    labels = ["1 step", f"2–{MAX_INTERP_GAP}", f"{MAX_INTERP_GAP + 1}–{10 * MAX_INTERP_GAP}", f"> {10 * MAX_INTERP_GAP}"]
    buckets = pd.cut(gaps["steps"], bins=bins, labels=labels).value_counts().reindex(labels)
    fig.add_trace(go.Bar(x=labels, y=buckets.values, marker_color=ORANGE, showlegend=False, text=buckets.values,
                         textposition="outside", hovertemplate="%{x}: %{y:,} gaps<extra></extra>"), row=1, col=2)
fig.update_yaxes(title_text="% observed", range=[0, 105], row=1, col=1)
fig.update_yaxes(title_text="Number of gaps", row=1, col=2)
show_fig(fig, f"{is_missing.mean():.1%} of steps are missing; longest gap = {gaps['steps'].max() if len(gaps) else 0:,} steps",
         height=420)

**Observation.** ✏️ Where the gaps are, how long. Decision: short gaps (≤ `MAX_INTERP_GAP`) are interpolated in 6.1; if a period has long gaps, restrict the study with `WINDOW_START` / `WINDOW_END` and justify it.

### 4.3 Missing values per column (after reindexing)

In [ ]:
na = regular.isna().sum()
pd.DataFrame({"missing steps": na, "% of steps": na / len(regular) * 100})

### 4.4 Impossible values

Physically impossible values (negative counts, 0 K temperature, rainfall above world records) are **measurement errors**: they become `NaN` in 6.1 and are then treated like missing steps.

In [ ]:
rows = []
for c, (low, high) in VALID_RANGES.items():
    too_low = int((regular[c] < low).sum()) if low is not None else 0
    too_high = int((regular[c] > high).sum()) if high is not None else 0
    rows.append({"column": c, "rule": f"[{low}, {high}]", "below": too_low, "above": too_high,
                 "observed min": regular[c].min(), "observed max": regular[c].max()})
pd.DataFrame(rows) if rows else print("VALID_RANGES is empty: add physical limits in section 1.3.")

### 4.5 Outliers of the target: robust z-score

A global z-score fails on seasonal data (night values look like outliers next to daytime values). So:
1. remove the **seasonal profile**: the median of all steps at the same position in the longest cycle (e.g. same hour of the week) → residual $r_t$;
2. compute a **robust** z-score in a rolling window of one cycle: $z_t = \dfrac{r_t - \text{median}(r)}{1.4826 \cdot \text{MAD}(r)}$. Median and MAD are not pulled by the outliers themselves (a rolling mean/std would be);
3. flag $|z_t| > 3.5$ (Iglewicz–Hoaglin threshold).

Flagged values are **kept** if they are real events (holiday, storm, closure) and only used to analyse errors in section 8.5.

In [ ]:
Z_THRESHOLD = 3.5
P = max(SEASONAL_PERIODS)
y_reg = regular[TARGET]
position = pd.Series(np.arange(len(y_reg)) % P, index=y_reg.index)     # position inside the longest cycle
profile = y_reg.groupby(position).transform("median")                  # usual value at that position
resid = y_reg - profile
window = resid.rolling(P, center=True, min_periods=P // 2)
mad = window.apply(lambda a: np.nanmedian(np.abs(a - np.nanmedian(a))), raw=True)
z_robust = (resid - window.median()) / (1.4826 * mad)
is_outlier = z_robust.abs() > Z_THRESHOLD

print(f"Outliers: {int(is_outlier.sum()):,} steps ({is_outlier.mean():.2%})")
fig = go.Figure()
fig.add_trace(go.Scattergl(x=y_reg.index, y=y_reg.values, mode="lines", line=dict(color=BLUE, width=1), name=TARGET))
fig.add_trace(go.Scattergl(x=y_reg.index[is_outlier.to_numpy()], y=y_reg[is_outlier], mode="markers", name="robust |z| > 3.5",
                           marker=dict(color=RED, size=6)))
show_fig(fig, f"{int(is_outlier.sum()):,} steps deviate strongly from the usual seasonal value", height=420,
         yaxis_title=TARGET, xaxis=dict(rangeslider=dict(visible=True, thickness=0.06)))

**Observation.** ✏️ When do outliers happen (dates)? Real events or errors? Decision (keep + flag, or treat as errors in `VALID_RANGES`).

<a id="s5"></a>
## 5. Exploratory data analysis (EDA)

EDA runs on `regular` (complete calendar, gaps left as `NaN`, so they show as breaks in line charts). Decomposition and ACF need a gap-free series and use a temporary interpolated copy.

### 5.1 Overview: level and long-term trend

In [ ]:
OVERVIEW_FREQ = "D"   # TODO: coarser period for the overview ("D" for hourly data, "W" or "MS" for daily data)
overview = y_reg.resample(OVERVIEW_FREQ).mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=overview.index, y=overview.values, mode="lines", line=dict(color=BLUE, width=1.2),
                         name=f"mean per {OVERVIEW_FREQ}"))
fig.add_trace(go.Scatter(x=overview.index, y=overview.rolling(30, center=True, min_periods=10).mean().values,
                         mode="lines", line=dict(color=ORANGE, width=2.5), name="30-period rolling mean"))
show_fig(fig, "✏️ <conclusion about trend / level shifts>", height=430, yaxis_title=TARGET, hovermode="x unified",
         xaxis=dict(rangeslider=dict(visible=True, thickness=0.06)))

**Observation.** ✏️ Trend, level shifts, structural breaks, periods to exclude.

### 5.2 A close-up: what does one cycle look like?

In [ ]:
ZOOM_STEPS = 2 * max(SEASONAL_PERIODS)  # two of the longest cycles
zoom = y_reg.iloc[-ZOOM_STEPS:]
fig = go.Figure(go.Scatter(x=zoom.index, y=zoom.values, mode="lines+markers", line=dict(color=BLUE, width=2),
                           marker=dict(size=4), hovertemplate="%{x|%a %Y-%m-%d %H:%M}: %{y:,.2f}<extra></extra>"))
show_fig(fig, f"Last {ZOOM_STEPS} steps: ✏️ <describe the repeating pattern>", height=400, yaxis_title=TARGET,
         showlegend=False)

### 5.3 Distribution of the target

In [ ]:
fig = go.Figure(hist_bar(y_reg, bins=60))
fig.add_vline(x=y_reg.median(), line_dash="dot", line_color="#52514e",
              annotation_text=f"median {y_reg.median():,.2f}", annotation_font_color="#52514e")
show_fig(fig, f"Distribution of {TARGET} (skewness {y_reg.skew():.2f})", xaxis_title=TARGET, yaxis_title="Steps",
         showlegend=False)
y_reg.describe().to_frame()

**Observation.** ✏️ Shape (skewed, bimodal?), range, consequence (e.g. MAPE is unstable near 0, a log transform could help).

### 5.4 Calendar effects

Average target by calendar position. The heatmap shows **hour × weekday** for intraday data, **month × weekday** otherwise; the box plot shows the spread per month.

In [ ]:
idx = y_reg.index
INTRADAY = idx.hour.nunique() > 1
DAY_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
cal = pd.DataFrame({"y": y_reg, "dow": idx.dayofweek, "month": idx.month, "hour": idx.hour})
columns = "hour" if INTRADAY else "month"
pivot = cal.pivot_table(index="dow", columns=columns, values="y", aggfunc="mean").reindex(range(7))

fig = go.Figure(go.Heatmap(z=pivot.values, x=pivot.columns, y=DAY_LABELS, colorscale=SEQ_BLUE, xgap=2, ygap=2,
                           colorbar=dict(title=TARGET), hovertemplate=f"%{{y}}, {columns} %{{x}}: %{{z:,.2f}}<extra></extra>"))
show_fig(fig, f"Mean {TARGET} by weekday x {columns}: ✏️ <conclusion>", height=420, xaxis_title=columns,
         yaxis_autorange="reversed")

fig = go.Figure([go.Box(y=cal.loc[cal["month"] == m, "y"], name=pd.Timestamp(2000, m, 1).strftime("%b"),
                        marker_color=BLUE, boxpoints=False, showlegend=False) for m in range(1, 13)])
show_fig(fig, f"{TARGET} by month: ✏️ <conclusion about yearly seasonality>", height=400, yaxis_title=TARGET)

**Observation.** ✏️ Which cycle dominates (daily, weekly, yearly)? Quote peak vs trough.

### 5.5 Seasonal decomposition

`MSTL` (or `STL` for a single cycle) splits the series into **trend + one seasonal component per period + residual** using LOESS smoothing. The share of variance of each component says what a model must capture. Periods longer than half the series are skipped (not estimable).

In [ ]:
y_filled = y_reg.interpolate(limit_direction="both")              # temporary gap-free copy, for decomposition only
periods = [p for p in SEASONAL_PERIODS if 2 * p < len(y_filled)]
if len(periods) > 1:
    decomposition = MSTL(y_filled, periods=periods).fit()
    seasonal = decomposition.seasonal                             # one column per period: seasonal_24, seasonal_168...
else:
    decomposition = STL(y_filled, period=periods[0]).fit()
    seasonal = decomposition.seasonal.to_frame(f"seasonal_{periods[0]}")

components = pd.concat([y_filled.rename("observed"), decomposition.trend.rename("trend"), seasonal,
                        decomposition.resid.rename("residual")], axis=1)
share = (components.drop(columns="observed").var() / components["observed"].var() * 100).rename("% of variance")
share.to_frame()

In [ ]:
recent = components.iloc[-3 * max(periods):]
fig = make_subplots(rows=len(recent.columns), cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=list(recent.columns))
for i, c in enumerate(recent.columns, start=1):
    fig.add_trace(go.Scatter(x=recent.index, y=recent[c], mode="lines", line=dict(color=QUALITATIVE[(i - 1) % 6], width=1.6),
                             showlegend=False, hovertemplate="%{x}: %{y:,.2f}<extra>" + c + "</extra>"), row=i, col=1)
show_fig(fig, f"Decomposition over the last {3 * max(periods)} steps: ✏️ <which component dominates>",
         height=160 * len(recent.columns) + 100)

**Observation.** ✏️ Variance share of trend / each season / residual → is differencing needed? Which cycles must the features encode?

### 5.6 Autocorrelation (ACF / PACF)

- **ACF** at lag $k$: correlation between $y_t$ and $y_{t-k}$. Peaks at the seasonal periods confirm the cycles.
- **PACF** at lag $k$: the same correlation after removing the effect of the lags in between — it points to lags that add new information.
- Bars outside the dotted band ($\pm 1.96/\sqrt{n}$) are significant. These plots choose `LAGS`.

In [ ]:
N_ACF = min(3 * max(SEASONAL_PERIODS), len(y_filled) // 4)
N_PACF = min(3 * min(SEASONAL_PERIODS), len(y_filled) // 4)
acf_vals = acf(y_filled, nlags=N_ACF, fft=True)
pacf_vals = pacf(y_filled, nlags=N_PACF, method="ywm")
band = 1.96 / np.sqrt(len(y_filled))

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14,
                    subplot_titles=(f"ACF up to lag {N_ACF}", f"PACF up to lag {N_PACF}"))
fig.add_trace(go.Bar(x=np.arange(N_ACF + 1), y=acf_vals, marker_color=BLUE, hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=np.arange(N_PACF + 1), y=pacf_vals, marker_color=ORANGE, hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=2, col=1)
for r in (1, 2):
    for s in (band, -band):
        fig.add_hline(y=s, line_dash="dot", line_color=MUTED, row=r, col=1)
for p in SEASONAL_PERIODS:
    if p <= N_ACF:
        fig.add_annotation(x=p, y=acf_vals[p], text=f"lag {p}: {acf_vals[p]:.2f}", showarrow=True, ay=-30,
                           arrowcolor=MUTED, font_color="#52514e", row=1, col=1)
fig.update_xaxes(title_text="Lag (steps)", row=2, col=1)
show_fig(fig, "Autocorrelation: ✏️ <which lags matter>", height=640, showlegend=False, bargap=0.3)
pd.DataFrame({"ACF": [acf_vals[k] if k <= N_ACF else np.nan for k in LAGS],
              "PACF": [pacf_vals[k] if k <= N_PACF else np.nan for k in LAGS]}, index=pd.Index(LAGS, name="lag"))

**Observation.** ✏️ Strongest lags with values → update `LAGS` in 1.3 if needed.

### 5.7 Covariates: do they move the target?

Raw correlations are confounded by the calendar (e.g. fog happens at night, when the target is low anyway). So each covariate is compared with the **deviation from the usual seasonal value** (`resid` from 4.5), not with the raw target.

In [ ]:
if NUM_EXOG:
    corr = pd.DataFrame({
        f"r with {TARGET}": [regular[c].corr(y_reg) for c in NUM_EXOG],
        "r with deviation from seasonal profile": [regular[c].corr(resid) for c in NUM_EXOG],
        f"r with deviation, covariate {HORIZON} step(s) earlier": [regular[c].shift(HORIZON).corr(resid) for c in NUM_EXOG],
    }, index=NUM_EXOG)
    display(corr)

for c in CAT_EXOG:
    effect = resid.groupby(regular[c]).agg(["mean", "count"])
    effect = effect[effect["count"] >= 30].sort_values("mean")   # ignore categories seen fewer than 30 times
    fig = go.Figure(go.Bar(x=effect["mean"], y=effect.index.astype(str), orientation="h",
                           marker_color=np.where(effect["mean"] < 0, RED, BLUE),
                           text=[f"n={n:,}" for n in effect["count"]], textposition="outside",
                           hovertemplate="%{y}: %{x:+,.2f}<extra></extra>"))
    fig.add_vline(x=0, line_color="#52514e", line_width=1)
    show_fig(fig, f"Mean deviation of {TARGET} from its seasonal profile, by {c}",
             xaxis_title="Mean deviation (target units)", height=max(360, 32 * len(effect)), showlegend=False)

**Observation.** ✏️ Which covariates matter after removing the calendar effect, with numbers.

### 5.8 Question 1: ✏️ <question, e.g. "How much lower is the target on holidays?">

Pattern for every question: **question → figure with a conclusion title → numbers → conclusion**. Rubric ideas: effect of an event type, comparison of two periods, forecast vs actual on a special day.

In [ ]:
# TODO: analysis for question 1 (compute -> figure with show_fig -> print the numbers quoted in the conclusion).

**Conclusion.** ✏️

### 5.9 Question 2: ✏️ <question>

In [ ]:
# TODO: analysis for question 2 (compute -> figure with show_fig -> print the numbers quoted in the conclusion).

**Conclusion.** ✏️

<a id="s6"></a>
## 6. Data preparation

### 6.1 Cleaning: regular calendar, errors to NaN, short gaps interpolated

1. merge duplicated timestamps and reindex at `FREQ` (`to_regular`, section 4.2);
2. keep `WINDOW_START`–`WINDOW_END`;
3. impossible values → `NaN`;
4. flag `is_imputed = 1` where the target was missing, then fill **only gaps of at most `MAX_INTERP_GAP` steps**: numeric columns by time-linear interpolation (the series changes continuously), categorical columns by forward fill (the last known state). Note that `interpolate(limit=n)` alone would also fill the first n steps of *longer* gaps; the mask below prevents that.

Longer gaps stay `NaN`: the rows whose lags fall into them are dropped after feature building.

In [ ]:
def short_gap_mask(s, max_len):
    """True on NaN cells that belong to a run of at most max_len consecutive NaN."""
    isna = s.isna()
    run_id = isna.ne(isna.shift()).cumsum()
    run_len = isna.groupby(run_id).transform("sum")
    return isna & (run_len <= max_len)


def clean_series(df):
    out = to_regular(df).loc[WINDOW_START:WINDOW_END]
    for c, (low, high) in VALID_RANGES.items():
        bad = (out[c] < (-np.inf if low is None else low)) | (out[c] > (np.inf if high is None else high))
        out.loc[bad, c] = np.nan
    out["is_imputed"] = out[TARGET].isna().astype(int)
    for c in [TARGET, *NUM_EXOG]:
        filled = out[c].interpolate(method="time", limit_area="inside")
        out[c] = out[c].where(~short_gap_mask(out[c], MAX_INTERP_GAP), filled)
    for c in CAT_EXOG:
        out[c] = out[c].where(~short_gap_mask(out[c], MAX_INTERP_GAP), out[c].ffill())
    out.loc[out[TARGET].isna(), "is_imputed"] = 0   # still missing (long gap): not imputed, simply absent
    return out


ts = clean_series(raw)
pd.Series({
    "steps in the study window": len(ts),
    "target interpolated (short gaps)": int(ts["is_imputed"].sum()),
    "target still missing (long gaps)": int(ts[TARGET].isna().sum()),
}, name="count").to_frame()

### 6.2 Feature engineering

Each row at time $t$ predicts $y_t$ using **only information available at $t - \text{HORIZON}$**:

| Group | Features | Leakage guard |
|---|---|---|
| Lags | `lag_k` = $y_{t-k}$ for k in `LAGS` | `LAGS` keeps k ≥ HORIZON |
| Rolling | mean / std of the target over `ROLL_WINDOWS` | computed on $y$ shifted by HORIZON |
| Calendar | hour, weekday, month as **sin/cos** pairs, `is_weekend` | known in advance |
| Covariates | value at $t - \text{HORIZON}$ | shifted by HORIZON |

Sin/cos encoding puts 23h next to 0h and December next to January, which a plain integer cannot.

> ✏️ Add domain features here (holidays, events) and document them in the table.

In [ ]:
def cyclical(values, period):
    angle = 2 * np.pi * np.asarray(values) / period
    return np.sin(angle), np.cos(angle)


def make_features(ts):
    """Feature table for every time step; the target column is kept for training."""
    f = pd.DataFrame(index=ts.index)
    y = ts[TARGET]
    for k in LAGS:
        f[f"lag_{k}"] = y.shift(k)                   # shift(k): value k rows earlier (rows = time steps)
    past = y.shift(HORIZON)
    for w in ROLL_WINDOWS:
        f[f"roll_mean_{w}"] = past.rolling(w).mean()
        f[f"roll_std_{w}"] = past.rolling(w).std()
    idx = ts.index
    if INTRADAY:
        f["hour_sin"], f["hour_cos"] = cyclical(idx.hour, 24)
    f["dow_sin"], f["dow_cos"] = cyclical(idx.dayofweek, 7)
    f["month_sin"], f["month_cos"] = cyclical(idx.month - 1, 12)
    f["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    for c in [*NUM_EXOG, *CAT_EXOG]:
        f[f"{c}_lag{HORIZON}"] = ts[c].shift(HORIZON)
    f[TARGET] = y
    return f


features = make_features(ts)
CAT_FEATURES = [f"{c}_lag{HORIZON}" for c in CAT_EXOG]
NUM_FEATURES = [c for c in features.columns if c not in [TARGET, *CAT_FEATURES]]
FEATURE_COLS = NUM_FEATURES + CAT_FEATURES

model_df = features.dropna(subset=[*NUM_FEATURES, TARGET])  # drops warm-up rows (first max(LAGS) steps) and long gaps
print(f"{len(features):,} steps -> {len(model_df):,} usable rows ({len(features) - len(model_df):,} dropped)")
model_df.head()

### 6.3 Chronological train / validation / test split

**No shuffling.** With a random split, `lag_1` of a test row would be the target of a training row, and the model would be scored on values it has effectively seen (leakage). Train is the past, validation the following period, test the most recent period.

In [ ]:
t = model_df.index
train = model_df[t <= pd.Timestamp(TRAIN_END)]
val = model_df[(t > pd.Timestamp(TRAIN_END)) & (t <= pd.Timestamp(VAL_END))]
test = model_df[t > pd.Timestamp(VAL_END)]
assert len(train) and len(val) and len(test), "TRAIN_END / VAL_END leave an empty set"
assert train.index.max() < val.index.min() and val.index.max() < test.index.min()

split_table = pd.DataFrame({
    "from": [p.index.min() for p in (train, val, test)],
    "to": [p.index.max() for p in (train, val, test)],
    "rows": [len(p) for p in (train, val, test)],
    "share": [len(p) / len(model_df) for p in (train, val, test)],
    f"mean {TARGET}": [p[TARGET].mean() for p in (train, val, test)],
    "% interpolated": [ts.loc[p.index, "is_imputed"].mean() * 100 for p in (train, val, test)],
}, index=["train", "validation", "test"])
display(split_table)

fig = go.Figure()
for name, part, color in [("train", train, BLUE), ("validation", val, ORANGE), ("test", test, AQUA)]:
    shown = part[TARGET].resample(OVERVIEW_FREQ).mean()
    fig.add_trace(go.Scatter(x=shown.index, y=shown.values, mode="lines", name=f"{name} ({len(part):,} rows)",
                             line=dict(color=color, width=1.6)))
show_fig(fig, "Chronological split: train -> validation -> test, no overlap", height=420, yaxis_title=TARGET,
         hovermode="x unified")

**Observation.** ✏️ Periods and seasons covered by each set; is the test period representative?

### 6.4 Scaling and encoding, fitted on train only

`StandardScaler` for numeric features (needed by Ridge; trees ignore it), `OneHotEncoder(handle_unknown="ignore")` for categorical covariates (a category first seen in test becomes all zeros instead of an error). Both live inside each model pipeline, so they are fitted on the training rows only — also inside `TimeSeriesSplit`.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), NUM_FEATURES),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT_FEATURES),
])
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val, y_val = val[FEATURE_COLS], val[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]

prep_check = clone(preprocessor).fit(X_train)
print(f"{len(FEATURE_COLS)} features -> {len(prep_check.get_feature_names_out())} model columns")
pd.DataFrame(prep_check.transform(X_train), columns=prep_check.get_feature_names_out()).describe().T[["mean", "std"]].head(12)

<a id="s7"></a>
## 7. Modeling

### 7.1 Baselines and models on validation

| Model | Why |
|---|---|
| **Naive** | forecast = last known value ($y_{t-\text{HORIZON}}$) |
| **Seasonal naive** | forecast = value one cycle earlier ($y_{t-P}$); often hard to beat on seasonal data |
| `Ridge` | linear combination of lags + calendar; L2 penalty keeps it stable with correlated lags |
| `RandomForestRegressor` | non-linear, averages many trees |
| `HistGradientBoostingRegressor` | boosted trees, strong on feature tables |

Metrics: **MAE** (average absolute error, in target units — main metric), **RMSE** (punishes large errors), **MAPE** (relative, undefined near 0 — computed on non-zero targets only), **R²**, and **skill** = $1 - \text{MAE}_\text{model} / \text{MAE}_\text{naive}$ (> 0 means better than naive). With `EVAL_ON_OBSERVED_ONLY`, interpolated steps are excluded from scoring.

In [ ]:
def score_mask(index):
    return (ts.loc[index, "is_imputed"] == 0).to_numpy() if EVAL_ON_OBSERVED_ONLY else np.ones(len(index), bool)


def regression_scores(y_true, y_pred):
    y_pred = np.asarray(y_pred, dtype=float)
    keep = score_mask(y_true.index) & ~np.isnan(y_pred)   # a baseline lag can fall into a long gap -> no forecast
    y_true, y_pred = y_true.to_numpy()[keep], y_pred[keep]
    err = y_true - y_pred
    nonzero = np.abs(y_true) > 1e-8
    return {"MAE": np.mean(np.abs(err)), "RMSE": np.sqrt(np.mean(err ** 2)),
            "MAPE %": np.mean(np.abs(err[nonzero] / y_true[nonzero])) * 100, "R2": r2_score(y_true, y_pred)}


BASELINES = {f"Naive (t-{HORIZON})": HORIZON,
             **{f"Seasonal naive (t-{p})": p for p in SEASONAL_PERIODS if p > HORIZON}}


def baseline_forecast(index, lag):
    return ts[TARGET].shift(lag).reindex(index).to_numpy()   # value `lag` steps before each target time


def make_model(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])


MODELS = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=SEED),
}

fitted, rows = {}, []
for name, lag in BASELINES.items():
    rows.append({"model": name, **regression_scores(y_val, baseline_forecast(y_val.index, lag))})
for name, estimator in MODELS.items():
    fitted[name] = make_model(estimator).fit(X_train, y_train)
    rows.append({"model": name, **regression_scores(y_val, fitted[name].predict(X_val))})
    print(f"done: {name}")

results = pd.DataFrame(rows).set_index("model")
results["skill vs naive"] = 1 - results["MAE"] / results.loc[f"Naive (t-{HORIZON})", "MAE"]
results.sort_values("MAE")

In [ ]:
ordered = results.sort_values("MAE", ascending=False)
fig = go.Figure(go.Bar(x=ordered["MAE"], y=ordered.index, orientation="h",
                       marker_color=[MUTED if n in BASELINES else BLUE for n in ordered.index],
                       text=[f"{m:,.2f}" for m in ordered["MAE"]], textposition="outside",
                       hovertemplate="%{y}: MAE %{x:,.3f}<extra></extra>"))
show_fig(fig, f"Validation MAE (grey = baselines): best = {results['MAE'].idxmin()}", xaxis_title=f"MAE ({TARGET} units)",
         height=380, showlegend=False)

**Observation.** ✏️ Ranking, skill over the naive baselines, linear vs tree models.

### 7.2 Hyperparameter tuning with `TimeSeriesSplit`

Ordinary k-fold would train on the future and test on the past. `TimeSeriesSplit` creates folds where the training part always **precedes** the test part (expanding window), on the training set only.

In [ ]:
TUNE_MODEL = "HistGradientBoosting"   # TODO: best ML model of 7.1
PARAM_GRID = {                        # TODO: adapt to TUNE_MODEL
    "model__learning_rate": [0.05, 0.1],
    "model__max_leaf_nodes": [15, 31, 63],
}
search = GridSearchCV(make_model(MODELS[TUNE_MODEL]), PARAM_GRID, scoring="neg_mean_absolute_error",
                      cv=TimeSeriesSplit(n_splits=3), n_jobs=-1)
search.fit(X_train, y_train)

tuned_name = f"{TUNE_MODEL} (tuned)"
fitted[tuned_name] = search.best_estimator_
results.loc[tuned_name] = pd.Series(regression_scores(y_val, search.best_estimator_.predict(X_val)))
results["skill vs naive"] = 1 - results["MAE"] / results.loc[f"Naive (t-{HORIZON})", "MAE"]
print("Best parameters:", search.best_params_, f"| CV MAE = {-search.best_score_:,.3f}")
results.sort_values("MAE")

### 7.3 Choose the final model

Best validation MAE among the ML models, refitted on train + validation (the most recent data matters most for forecasting).

In [ ]:
ml_results = results.drop(index=list(BASELINES))
BEST_NAME = ml_results["MAE"].idxmin()
final_model = clone(fitted[BEST_NAME]).fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
print(f"Final model: {BEST_NAME} (validation MAE {results.loc[BEST_NAME, 'MAE']:,.3f}, "
      f"skill vs naive {results.loc[BEST_NAME, 'skill vs naive']:+.1%})")

<a id="s8"></a>
## 8. Evaluation on the test set

### 8.1 Test metrics

In [ ]:
y_pred = pd.Series(final_model.predict(X_test), index=y_test.index)
rows = [{"model": name, **regression_scores(y_test, baseline_forecast(y_test.index, lag))} for name, lag in BASELINES.items()]
rows.append({"model": f"Final: {BEST_NAME}", **regression_scores(y_test, y_pred)})
test_scores = pd.DataFrame(rows).set_index("model")
test_scores["skill vs naive"] = 1 - test_scores["MAE"] / test_scores.loc[f"Naive (t-{HORIZON})", "MAE"]
test_scores

### 8.2 Forecast vs actual

In [ ]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12,
                    subplot_titles=(f"Whole test period (mean per {OVERVIEW_FREQ})", f"Last {ZOOM_STEPS} steps"))
for name, series, color, dash in [("actual", y_test, BLUE, None), ("forecast", y_pred, ORANGE, None)]:
    coarse = series.resample(OVERVIEW_FREQ).mean()
    fig.add_trace(go.Scatter(x=coarse.index, y=coarse.values, mode="lines", name=name, legendgroup=name,
                             line=dict(color=color, width=1.6)), row=1, col=1)
    fine = series.iloc[-ZOOM_STEPS:]
    fig.add_trace(go.Scatter(x=fine.index, y=fine.values, mode="lines", name=name, legendgroup=name, showlegend=False,
                             line=dict(color=color, width=2)), row=2, col=1)
naive = pd.Series(baseline_forecast(y_test.index, HORIZON), index=y_test.index).iloc[-ZOOM_STEPS:]
fig.add_trace(go.Scatter(x=naive.index, y=naive.values, mode="lines", name=f"naive (t-{HORIZON})",
                         line=dict(color=MUTED, width=1.2, dash="dot")), row=2, col=1)
show_fig(fig, f"Test forecast vs actual: MAE {test_scores.iloc[-1]['MAE']:,.2f} "
              f"({test_scores.iloc[-1]['skill vs naive']:+.1%} vs naive)", height=700, hovermode="x unified")

**Observation.** ✏️ Where the forecast follows well, where it misses (peaks, sudden drops).

### 8.3 Where are the errors?

In [ ]:
err = (y_test - y_pred)[score_mask(y_test.index)]
group = err.index.hour if INTRADAY else err.index.dayofweek
group_name = "hour" if INTRADAY else "weekday"
mae_by_group = err.abs().groupby(group).mean()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Residual distribution (actual - forecast)", f"MAE by {group_name}"))
fig.add_trace(hist_bar(err, bins=60), row=1, col=1)
fig.add_trace(go.Bar(x=mae_by_group.index, y=mae_by_group.values, marker_color=ORANGE,
                     hovertemplate=f"{group_name} %{{x}}: MAE %{{y:,.2f}}<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text=f"Error ({TARGET} units)", row=1, col=1)
fig.update_xaxes(title_text=group_name, row=1, col=2)
show_fig(fig, f"Residuals: mean {err.mean():+,.2f} (bias), largest MAE at {group_name} {mae_by_group.idxmax()}",
         height=420, showlegend=False)

### 8.4 Which features matter?

Permutation importance on test: shuffle one feature, measure the increase in MAE.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="neg_mean_absolute_error", n_repeats=5,
                              random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({"MAE increase": perm.importances_mean, "std": perm.importances_std}, index=FEATURE_COLS)
                .sort_values("MAE increase").tail(15))
fig = go.Figure(go.Bar(x=importance["MAE increase"], y=importance.index, orientation="h", marker_color=BLUE,
                       error_x=dict(type="data", array=importance["std"], color=MUTED)))
show_fig(fig, f"Permutation importance on test: '{importance.index[-1]}' matters most",
         xaxis_title="Increase in MAE when the feature is shuffled", height=max(380, 30 * len(importance)))

**Observation.** ✏️ Consistent with the ACF (5.6) and covariate analysis (5.7)?

### 8.5 Largest errors

Do the worst errors coincide with the outliers flagged in 4.5 (real but unpredictable events)?

In [ ]:
worst = pd.DataFrame({"actual": y_test, "forecast": y_pred, "abs error": (y_test - y_pred).abs(),
                      "flagged outlier (4.5)": is_outlier.reindex(y_test.index, fill_value=False).astype(bool)})
share_flagged = worst.nlargest(50, "abs error")["flagged outlier (4.5)"].mean()
print(f"{share_flagged:.0%} of the 50 largest errors fall on steps flagged as outliers "
      f"(vs {worst['flagged outlier (4.5)'].mean():.1%} of all test steps)")
worst.nlargest(10, "abs error")

**Observation.** ✏️

<a id="s9"></a>
## 9. Inference on new data

For a time series, "new data" means **the future**. After evaluation, the model is refitted on **all** usable history (train + validation + test), then forecasts the next `HORIZON` steps after the last observation. The future rows get their features from `make_features`: lags ≥ HORIZON, rolling windows and covariates shifted by HORIZON all point to already observed steps; calendar features are known in advance.

In [ ]:
deploy_model = clone(final_model).fit(model_df[FEATURE_COLS], model_df[TARGET])

last_time = ts.index.max()
future_index = pd.date_range(last_time, periods=HORIZON + 1, freq=FREQ)[1:]
extended = pd.concat([ts, pd.DataFrame(index=future_index)])     # future rows: target and covariates unknown (NaN)
future_X = make_features(extended).loc[future_index, FEATURE_COLS]
forecast = pd.Series(deploy_model.predict(future_X), index=future_index, name="forecast")

history = ts[TARGET].iloc[-ZOOM_STEPS:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=history.index, y=history.values, mode="lines", name="history", line=dict(color=BLUE, width=2)))
fig.add_trace(go.Scatter(x=forecast.index, y=forecast.values, mode="lines+markers", name="forecast",
                         line=dict(color=ORANGE, width=2, dash="dash"), marker=dict(size=8)))
show_fig(fig, f"Forecast for the next {HORIZON} step(s) after {last_time}", height=420, yaxis_title=TARGET,
         hovermode="x unified")
forecast.to_frame()

<a id="s10"></a>
## 10. Conclusion

### 10.1 Summary of results

✏️ Fill this table with the numbers printed in sections 7–8; the same numbers must appear in the PDF report and on the project page (`assets/js/data.js → results`).

| Model | MAE | RMSE | MAPE % | Skill vs naive |
|---|---|---|---|---|
| Naive (t-HORIZON) | ✏️ | ✏️ | ✏️ | 0 |
| Seasonal naive | ✏️ | ✏️ | ✏️ | ✏️ |
| ✏️ final model | ✏️ | ✏️ | ✏️ | ✏️ |

### 10.2 What the analysis answered

✏️ 3–5 numbered findings, each with a number from this notebook and a link to its figure (e.g. "Figure 7 shows …").

### 10.3 Limitations

✏️ Data (size, period, sampling bias, label noise), method (assumptions, features not used), evaluation (single split, test period/distribution).

### 10.4 Extensions

✏️ Only list what you would actually try next. Ideas that were run and compared earn creativity points; ideas that were not run do not.

### 10.5 Submission checklist — delete before submitting

- [ ] **Runtime → Restart and run all** on Google Colab finishes without errors, and the saved notebook keeps its outputs.
- [ ] No `TODO`, `✏️` or placeholder text is left (search the notebook for `TODO`).
- [ ] Every figure is Plotly, has a numbered conclusion-style title and labelled axes, and is followed by an observation that quotes numbers.
- [ ] Every number written in markdown matches the printed output of the current run.
- [ ] Imputers, scalers, encoders and vectorizers are fitted on **train only**; the test set is used once, in section 8.
- [ ] The model comparison includes a baseline, and the chosen model is selected on **validation**, not test.
- [ ] The split is chronological; no feature uses information younger than `HORIZON` steps.
- [ ] `LAGS` and `SEASONAL_PERIODS` are justified by the ACF/PACF and decomposition in section 5.
- [ ] Section 5.8 contains at least one question-driven analysis with a forecast-vs-actual or before/after comparison.
- [ ] `assets/js/data.js` is updated: notebook path, dataset, problem, results.